In [22]:
from dotenv import load_dotenv
import os
from neo4j import GraphDatabase

load_dotenv(override=True)

AURA_INSTANCENAME = os.environ["AURA_INSTANCENAME"]
NEO4J_URI = os.environ["NEO4J_URI"]
NEO4J_USERNAME = os.environ["NEO4J_USERNAME"]
NEO4J_PASSWORD = os.environ["NEO4J_PASSWORD"]
AUTH = (NEO4J_USERNAME, NEO4J_PASSWORD)

print(AURA_INSTANCENAME)
print(NEO4J_URI)
print(NEO4J_USERNAME)
print(NEO4J_PASSWORD)
print(AUTH)

Instance02
neo4j+ssc://8c816eb4.databases.neo4j.io
neo4j
SjFH0CBSvP6ngGW2sACh6cvKzC8HNU2zN0zFJBEg1kE
('neo4j', 'SjFH0CBSvP6ngGW2sACh6cvKzC8HNU2zN0zFJBEg1kE')


In [23]:
driver = GraphDatabase.driver(NEO4J_URI, auth=AUTH)

In [24]:
def display_node_count():
    driver = GraphDatabase.driver(NEO4J_URI, auth=AUTH)
    try:
        with driver.session() as session: 
            result = session.run("MATCH (n) RETURN count(n) AS count")
            for record in result:
                print(record["count"])
    except Exception as e:
        print(f"Error: {e}")
    finally:
        driver.close()

In [25]:
print ("Node count before building the knowledge graph:")
display_node_count()

Node count before building the knowledge graph:
0


In [ ]:
def connect_and_query():
    try:
        with driver.session() as session:
            result = session.run("MATCH (n) RETURN count(n)")
            count = result.single().value()
            print(f"Number of nodes: {count}")
    except Exception as e:
        print(f"Error: {e}")
    finally:
        driver.close()

def create_entities(tx):
    # Create Albert Einstein node
    tx.run("MERGE (a:Person {name: 'Albert Einstein'})")

    # Create other nodes
    tx.run("MERGE (p:Subject {name: 'Physics'})")
    tx.run("MERGE (n:NobelPrize {name: 'Nobel Prize in Physics'})")
    tx.run("MERGE (g:Country {name: 'Germany'})")
    tx.run("MERGE (u:Country {name: 'USA'})")


def create_relationships(tx):
    # Create studied relationship
    tx.run(
        """
    MATCH (a:Person {name: 'Albert Einstein'}), (p:Subject {name: 'Physics'})
    MERGE (a)-[:STUDIED]->(p)
    """
    )

    # Create won relationship
    tx.run(
        """
    MATCH (a:Person {name: 'Albert Einstein'}), (n:NobelPrize {name: 'Nobel Prize in Physics'})
    MERGE (a)-[:WON]->(n)
    """
    )

    # Create born in relationship
    tx.run(
        """
    MATCH (a:Person {name: 'Albert Einstein'}), (g:Country {name: 'Germany'})
    MERGE (a)-[:BORN_IN]->(g)
    """
    )

    # Create died in relationship
    tx.run(
        """
    MATCH (a:Person {name: 'Albert Einstein'}), (u:Country {name: 'USA'})
    MERGE (a)-[:DIED_IN]->(u)
    """
    )


# Function to connect and run a simple Cypher query
def query_graph_simple(cypher_query):
    driver = GraphDatabase.driver(NEO4J_URI, auth=AUTH)
    try:
        with driver.session() as session:
            result = session.run(cypher_query)
            for record in result:
                print(record["name"])
    except Exception as e:
        print(f"Error: {e}")
    finally:
        driver.close()


# Function to connect and run a Cypher query
def query_graph(cypher_query):
    driver = GraphDatabase.driver(NEO4J_URI, auth=AUTH)
    try:
        with driver.session() as session: 
            result = session.run(cypher_query)
            for record in result:
                print(record["path"])
    except Exception as e:
        print(f"Error: {e}")
    finally:
        driver.close()


def build_knowledge_graph():
    # Open a session with the Neo4j database

    try:
        with driver.session() as session: #database=NEO4J_DATABASE
            # Create entities
            session.execute_write(create_entities)
            # Create relationships
            session.execute_write(create_relationships)

    except Exception as e:
        print(f"Error: {e}")
    finally:
        driver.close()


In [28]:
# if __name__ == "__main__":
#     build_knowledge_graph()

# Cypher query to find paths related to Albert Einstein
einstein_query = """
MATCH path=(a:Person {name: 'Albert Einstein'})-[:STUDIED]->(s:Subject)
RETURN path
UNION
MATCH path=(a:Person {name: 'Albert Einstein'})-[:WON]->(n:NobelPrize)
RETURN path
UNION
MATCH path=(a:Person {name: 'Albert Einstein'})-[:BORN_IN]->(g:Country)
RETURN path
UNION
MATCH path=(a:Person {name: 'Albert Einstein'})-[:DIED_IN]->(u:Country)
RETURN path
"""

# Simple Cypher query to find all node names
simple_query = """
MATCH (n)
RETURN n.name AS name
"""

In [ ]:
if __name__ == "__main__":
    # Build the knowledge graph
    build_knowledge_graph()

    query_graph_simple(
        simple_query
    )
    query_graph(einstein_query)

C:\Users\g4dqxc\AppData\Local\Temp\ipykernel_2908\1907977906.py:89: DeprecationWarning: Using a driver after it has been closed is deprecated. Future versions of the driver will raise an error.
  with driver.session() as session: #database=NEO4J_DATABASE


Albert Einstein
Physics
Nobel Prize in Physics
Germany
USA
<Path start=<Node element_id='4:5f6b4eec-e38d-45af-9ccd-664b3d1842fb:0' labels=frozenset({'Person'}) properties={'name': 'Albert Einstein'}> end=<Node element_id='4:5f6b4eec-e38d-45af-9ccd-664b3d1842fb:1' labels=frozenset({'Subject'}) properties={'name': 'Physics'}> size=1>
<Path start=<Node element_id='4:5f6b4eec-e38d-45af-9ccd-664b3d1842fb:0' labels=frozenset({'Person'}) properties={'name': 'Albert Einstein'}> end=<Node element_id='4:5f6b4eec-e38d-45af-9ccd-664b3d1842fb:2' labels=frozenset({'NobelPrize'}) properties={'name': 'Nobel Prize in Physics'}> size=1>
<Path start=<Node element_id='4:5f6b4eec-e38d-45af-9ccd-664b3d1842fb:0' labels=frozenset({'Person'}) properties={'name': 'Albert Einstein'}> end=<Node element_id='4:5f6b4eec-e38d-45af-9ccd-664b3d1842fb:3' labels=frozenset({'Country'}) properties={'name': 'Germany'}> size=1>
<Path start=<Node element_id='4:5f6b4eec-e38d-45af-9ccd-664b3d1842fb:0' labels=frozenset({'Person'

In [ ]:
# Run this to see the entire graph in the neo4j browser/console
# MATCH (n)-[r]->(m)
# RETURN n, r, m;